# 03 — Modelado
## FraudShield-ML

Entrenamos y comparamos multiples modelos de clasificacion para deteccion de fraude.

### Objetivos
- Entrenar 6 modelos baseline con tracking en MLflow
- Comparar metricas AUC-PR, AUC-ROC, F1 en validacion
- Identificar el mejor modelo para tuning en el siguiente notebook

### Metrica principal
AUC-PR (Area bajo la curva Precision-Recall) — mas adecuada que AUC-ROC
cuando hay desbalance extremo de clases.

## 1. Importaciones

In [5]:
import warnings
warnings.filterwarnings('ignore')

import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import mlflow
import mlflow.sklearn

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from imblearn.ensemble import BalancedRandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    f1_score
)

print("Librerías cargadas")

Librerías cargadas


## 2. Carga de datos y configuracion de rutas

In [8]:
PROJECT_ROOT   = Path(".").resolve().parent
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
MODELS_DIR     = PROJECT_ROOT / "models"

X_train = pd.read_parquet(DATA_PROCESSED / "X_train.parquet")
X_val   = pd.read_parquet(DATA_PROCESSED / "X_val.parquet")
X_test  = pd.read_parquet(DATA_PROCESSED / "X_test.parquet")

y_train = pd.read_parquet(DATA_PROCESSED / "y_train.parquet").squeeze()
y_val   = pd.read_parquet(DATA_PROCESSED / "y_val.parquet").squeeze()
y_test  = pd.read_parquet(DATA_PROCESSED / "y_test.parquet").squeeze()

print(f"X_train: {X_train.shape}  |  fraude: {y_train.mean()*100:.2f}%")
print(f"X_val:   {X_val.shape}  |  fraude: {y_val.mean()*100:.2f}%")
print(f"X_test:  {X_test.shape}  |  fraude: {y_test.mean()*100:.2f}%")

X_train: (413378, 422)  |  fraude: 3.50%
X_val:   (88581, 422)  |  fraude: 3.50%
X_test:  (88581, 422)  |  fraude: 3.50%


## 3. Configuracion de MLflow

Usamos MLflow para trackear hiperparametros, metricas y artefactos de cada modelo.
Permite comparar experimentos y reproducir resultados.

In [9]:
mlflow.set_tracking_uri("file:///" + str(PROJECT_ROOT / "mlruns").replace("\\", "/"))
mlflow.set_experiment("fraudshield-models")

print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experimento:  fraudshield-models")

Tracking URI: file:///C:/Users/micke/OneDrive/Desktop/projects/Frauddetection-mlops/mlruns
Experimento:  fraudshield-models


## 4. Definicion de modelos

Definimos 6 modelos con configuracion inicial para comparacion baseline.
Incluimos modelos lineales, basados en arboles y ensemble boosting.

In [10]:
models = {
    "LogisticRegression": LogisticRegression(
        max_iter=1000,
        random_state=42,
        n_jobs=-1
    ),
    "DecisionTree": DecisionTreeClassifier(
        random_state=42
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),
    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),
    "BalancedRandomForest": BalancedRandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss',
        verbosity=0
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    ),
    "CatBoost": CatBoostClassifier(
        n_estimators=100,
        random_state=42,
        verbose=0
    )
}

print(f"{len(models)} modelos definidos:")
for name in models:
    print(f"  {name}")

8 modelos definidos:
  LogisticRegression
  DecisionTree
  RandomForest
  ExtraTrees
  BalancedRandomForest
  XGBoost
  LightGBM
  CatBoost


## 5. Entrenamiento y evaluacion de todos los modelos

Entrenamos cada modelo, calculamos metricas en train y validacion,
y registramos todo en MLflow.

In [11]:
results = []

for name, model in models.items():
    print(f"\nEntrenando {name}...")

    with mlflow.start_run(run_name=name):

        # Entrenar y medir tiempo
        t0 = time.time()
        model.fit(X_train, y_train)
        train_time = time.time() - t0

        # Predicciones
        proba_train = model.predict_proba(X_train)[:, 1]
        proba_val   = model.predict_proba(X_val)[:, 1]
        pred_val    = model.predict(X_val)

        # Métricas
        auc_pr_train = average_precision_score(y_train, proba_train)
        auc_pr_val   = average_precision_score(y_val,   proba_val)
        roc_auc_val  = roc_auc_score(y_val, proba_val)
        f1_val       = f1_score(y_val, pred_val)
        overfit_gap  = auc_pr_train - auc_pr_val

        # Loguear en MLflow
        mlflow.log_param("model_type",   name)
        mlflow.log_metric("auc_pr_train", auc_pr_train)
        mlflow.log_metric("auc_pr_val",   auc_pr_val)
        mlflow.log_metric("roc_auc_val",  roc_auc_val)
        mlflow.log_metric("f1_val",       f1_val)
        mlflow.log_metric("overfit_gap",  overfit_gap)
        mlflow.log_metric("train_time",   train_time)
        mlflow.sklearn.log_model(model, "model")

        results.append({
            "Modelo":       name,
            "AUC-PR Train": round(auc_pr_train, 4),
            "AUC-PR Val":   round(auc_pr_val,   4),
            "ROC-AUC Val":  round(roc_auc_val,  4),
            "F1 Val":       round(f1_val,        4),
            "Overfit Gap":  round(overfit_gap,   4),
            "Tiempo (s)":   round(train_time,    1)
        })

        print(f"  AUC-PR Val: {auc_pr_val:.4f} | ROC-AUC: {roc_auc_val:.4f} | Gap: {overfit_gap:.4f} | {train_time:.1f}s")

print("\nChallenge completado")


Entrenando LogisticRegression...


2026/03/25 21:28:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/25 21:28:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  AUC-PR Val: 0.4752 | ROC-AUC: 0.8642 | Gap: 0.0129 | 57.5s

Entrenando DecisionTree...


2026/03/25 21:30:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/25 21:30:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  AUC-PR Val: 0.3232 | ROC-AUC: 0.7834 | Gap: 0.6768 | 114.4s

Entrenando RandomForest...


...

  AUC-PR Val: 0.6937 | ROC-AUC: 0.9339 | Gap: 0.0836 | 13.8s

Entrenando LightGBM...


2026/03/25 21:34:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/25 21:34:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  AUC-PR Val: 0.6709 | ROC-AUC: 0.9288 | Gap: 0.0567 | 11.9s

Entrenando CatBoost...


2026/03/25 21:35:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/25 21:35:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  AUC-PR Val: 0.6451 | ROC-AUC: 0.9095 | Gap: 0.0497 | 16.4s

Challenge completado


## 6. Tabla comparativa de resultados

Comparamos todos los modelos por AUC-PR en validacion para seleccionar
el candidato al tuning de hiperparametros.

In [17]:
df_results = pd.DataFrame(results)
df_results = df_results.sort_values("AUC-PR Val", ascending=False).reset_index(drop=True)

print("=" * 75)
print("RESULTADOS DEL CHALLENGE")
print("=" * 75)
print(df_results.to_string(index=False))
print("=" * 75)

RESULTADOS DEL CHALLENGE
              Modelo  AUC-PR Train  AUC-PR Val  ROC-AUC Val  F1 Val  Overfit Gap  Tiempo (s)
          ExtraTrees        1.0000      0.7248       0.9285  0.6254       0.2752        72.3
        RandomForest        1.0000      0.7052       0.9249  0.5987       0.2948        48.9
             XGBoost        0.7773      0.6937       0.9339  0.6294       0.0836        13.8
            LightGBM        0.7276      0.6709       0.9288  0.5839       0.0567        11.9
            CatBoost        0.6948      0.6451       0.9095  0.6003       0.0497        16.4
BalancedRandomForest        0.9422      0.6323       0.9250  0.4149       0.3099        10.8
  LogisticRegression        0.4881      0.4752       0.8642  0.4273       0.0129        57.5
        DecisionTree        1.0000      0.3232       0.7834  0.5548       0.6768       114.4
